# Lab 1: Introduction to Machine Learning for Cybersecurity Attack Detection

## Learning Objectives
By the end of this lab, students will be able to:
- Understand the fundamentals of machine learning in cybersecurity contexts
- Implement basic ML algorithms for network intrusion detection
- Evaluate model performance using appropriate metrics
- Interpret results and understand the challenges of ML in cybersecurity

## Prerequisites
- Basic Python programming knowledge
- Understanding of network protocols (TCP/IP, HTTP)
- Familiarity with pandas and numpy libraries
- Google Colab account (free)

## Lab Setup for Google Colab
This lab is designed to run on Google Colab. All required libraries are pre-installed or will be installed automatically. The dataset will be downloaded directly in the notebook.

## Lab Overview
This lab introduces students to machine learning techniques for detecting cybersecurity attacks using the NSL-KDD dataset, a popular benchmark dataset for network intrusion detection research.

## Part 1: Understanding the Problem Domain

### 1.1 What is Network Intrusion Detection?
Network intrusion detection systems (NIDS) monitor network traffic to identify malicious activities or policy violations. Traditional signature-based systems rely on known attack patterns, but machine learning enables detection of previously unknown attacks (zero-day attacks).

### 1.2 Common Attack Types
- **DoS (Denial of Service)**: Overwhelming system resources
- **Probe**: Scanning for vulnerabilities
- **R2L (Remote to Local)**: Unauthorized access from remote machine
- **U2R (User to Root)**: Unauthorized access to root privileges

## Part 2: Dataset Exploration

### 2.1 NSL-KDD Dataset
The NSL-KDD dataset contains network connection records with 41 features and binary/multi-class labels indicating normal traffic or specific attack types.

**Key Features Include:**
- Duration of connection
- Protocol type (TCP, UDP, ICMP)
- Service (HTTP, FTP, SMTP, etc.)
- Number of bytes sent/received
- Connection state flags



### 2.2 Data Loading and Initial Exploration
#### Install required packages (if not already installed)


In [ ]:
!pip install -q pandas numpy matplotlib seaborn scikit-learn

#### Download the NSL-KDD dataset

In [ ]:
import os
import urllib.request

# Create data directory
os.makedirs('data', exist_ok=True)

# Download dataset files
base_url = "https://raw.githubusercontent.com/defcom17/NSL_KDD/master/"
files = ['KDDTrain+.txt', 'KDDTest+.txt']

for file in files:
    if not os.path.exists(f'data/{file}'):
        print(f"Downloading {file}...")
        urllib.request.urlretrieve(f"{base_url}{file}", f'data/{file}')
        print(f"Downloaded {file}")
    else:
        print(f"{file} already exists")

#### Import necessary libraries

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

#### Load the dataset

In [ ]:
# Load the dataset
print("Loading dataset...")
train_data = pd.read_csv('data/KDDTrain+.txt', header=None)
test_data = pd.read_csv('data/KDDTest+.txt', header=None)


####Define column names

In [ ]:
# Define column names
columns = ['duration', 'protocol_type', 'service', 'flag', 'src_bytes', 'dst_bytes',
           'land', 'wrong_fragment', 'urgent', 'hot', 'num_failed_logins', 'logged_in',
           'num_compromised', 'root_shell', 'su_attempted', 'num_root', 'num_file_creations',
           'num_shells', 'num_access_files', 'num_outbound_cmds', 'is_host_login',
           'is_guest_login', 'count', 'srv_count', 'serror_rate', 'srv_serror_rate',
           'rerror_rate', 'srv_rerror_rate', 'same_srv_rate', 'diff_srv_rate',
           'srv_diff_host_rate', 'dst_host_count', 'dst_host_srv_count',
           'dst_host_same_srv_rate', 'dst_host_diff_srv_rate', 'dst_host_same_src_port_rate',
           'dst_host_srv_diff_host_rate', 'dst_host_serror_rate', 'dst_host_srv_serror_rate',
           'dst_host_rerror_rate', 'dst_host_srv_rerror_rate', 'attack_type', 'difficulty']

train_data.columns = columns
test_data.columns = columns

print("Dataset loaded successfully!")
print(f"Training data shape: {train_data.shape}")
print(f"Test data shape: {test_data.shape}")

### 2.3 Exploratory Data Analysis

#### Display basic information

In [ ]:
# Display basic information
print("\nFirst few rows:")
print(train_data.head())

print("\nAttack type distribution:")
print(train_data['attack_type'].value_counts())

#### Create binary classification (Normal vs Attack)

In [ ]:
# Create binary classification (Normal vs Attack)
train_data['binary_class'] = train_data['attack_type'].apply(lambda x: 0 if x == 'normal' else 1)
test_data['binary_class'] = test_data['attack_type'].apply(lambda x: 0 if x == 'normal' else 1)

print("\nBinary classification distribution:")
print(train_data['binary_class'].value_counts())

#### Visualize attack distribution

In [ ]:
plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
train_data['binary_class'].value_counts().plot(kind='bar')
plt.title('Normal vs Attack Distribution')
plt.xticks([0, 1], ['Normal', 'Attack'], rotation=0)

plt.subplot(1, 2, 2)
top_attacks = train_data['attack_type'].value_counts().head(10)
top_attacks.plot(kind='bar')
plt.title('Top 10 Attack Types')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

**Question 1**: Give the short description for each attack type

## Part 3: Data Preprocessing

### 3.1 Handle Categorical Variables

#### Identify categorical and numerical columns


In [ ]:
# Identify categorical and numerical columns
categorical_cols = ['protocol_type', 'service', 'flag']
numerical_cols = [col for col in train_data.columns if col not in categorical_cols + ['attack_type', 'binary_class', 'difficulty']]

print(f"Categorical columns: {categorical_cols}")
print(f"Numerical columns: {len(numerical_cols)} columns")

#### Encode categorical variables


In [ ]:
label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    train_data[col] = le.fit_transform(train_data[col])
    test_data[col] = le.transform(test_data[col])
    label_encoders[col] = le

print("\nCategorical encoding completed")

### Explaination: Handling Categorical Variables in Machine Learning

**What are Categorical Variables?**

Categorical variables are features that contain text or string values representing categories or groups, rather than numerical measurements. In the NSL-KDD cybersecurity dataset, we have three main categorical variables:

1. **`protocol_type`**: The network protocol used (TCP, UDP, ICMP)
2. **`service`**: The network service accessed (HTTP, FTP, SMTP, telnet, etc.)
3. **`flag`**: Connection status flags (SF, S0, REJ, etc.)

**Why Do We Need to Handle Them?**

Machine learning algorithms are mathematical models that work with numbers, not text. They can't directly process strings like "TCP" or "HTTP". We need to convert these text values into numerical representations.

**The Process in Detail**

Let me break down what happens in this code section:

```python
# Identify categorical and numerical columns
categorical_cols = ['protocol_type', 'service', 'flag']
numerical_cols = [col for col in train_data.columns if col not in categorical_cols + ['attack_type', 'binary_class', 'difficulty']]
```

**What this does:**
- Explicitly defines which columns contain categorical data
- Creates a list of numerical columns by excluding categorical and target columns
- This separation helps us apply different preprocessing techniques to different data types

**Label Encoding Process**

```python
# Encode categorical variables
label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    train_data[col] = le.fit_transform(train_data[col])
    test_data[col] = le.transform(test_data[col])
    label_encoders[col] = le
```

***What Label Encoding Does:***

Let's say we have these values in `protocol_type`:
- **Original:** `['TCP', 'UDP', 'ICMP', 'TCP', 'UDP']`
- **After encoding:** `[2, 1, 0, 2, 1]`

The LabelEncoder assigns a unique integer to each unique string:
- `'ICMP'` → `0`
- `'UDP'` → `1`  
- `'TCP'` → `2`

***Why We Store the Encoders***

```python
label_encoders[col] = le
```

We save each encoder because:
1. **Consistency**: We need to apply the same mapping to test data
2. **Future Use**: When new data comes in, we need to encode it the same way
3. **Reversibility**: We can decode numbers back to original strings if needed

***Important Considerations***

###### 1. **Fit vs Transform**
- **`fit_transform()`** on training data: Learns the mapping AND applies it
- **`transform()`** on test data: Only applies the learned mapping
- This prevents data leakage from test set into training

###### 2. **Order Matters**
Label encoding creates an implicit ordering (0 < 1 < 2), which might not reflect real relationships. For example:
- TCP=2, UDP=1, ICMP=0 suggests TCP > UDP > ICMP
- But protocols don't have a natural order

###### 3. **Alternative Approaches**

**One-Hot Encoding** (alternative method):
```python
# Alternative: One-hot encoding
pd.get_dummies(data['protocol_type'])
```
This creates separate binary columns:
- `protocol_TCP`: `[1, 0, 0, 1, 0]`
- `protocol_UDP`: `[0, 1, 0, 0, 1]`  
- `protocol_ICMP`: `[0, 0, 1, 0, 0]`

**When to use each:**
- **Label Encoding**: For tree-based models (Random Forest, Decision Trees) - they can handle the artificial ordering
- **One-Hot Encoding**: For linear models (Logistic Regression) - avoids false ordering assumptions

##### Real Example from the Dataset

Let's see what actually happens:

**Before encoding:**
```
protocol_type: ['tcp', 'udp', 'icmp', 'tcp', 'tcp']
service: ['http', 'ftp', 'smtp', 'http', 'telnet']
flag: ['SF', 'S0', 'REJ', 'SF', 'SF']
```

**After encoding:**
```
protocol_type: [2, 1, 0, 2, 2]
service: [5, 3, 8, 5, 9]
flag: [1, 2, 0, 1, 1]
```

**Why This Matters for Cybersecurity**

1. **Feature Importance**: After encoding, the model can learn that certain protocols or services are more associated with attacks
2. **Pattern Recognition**: Numerical encoding allows algorithms to find patterns like "TCP + HTTP + SF flag = likely normal traffic"
3. **Scalability**: Encoded data processes much faster than string comparisons

**Checking the Results**

The code includes verification:
```python
print("Categorical encoding completed")
print("\nEncoded categorical columns:")
for col in categorical_cols:
    print(f"{col}: {train_data[col].unique()[:10]}")
```

This shows the unique encoded values, helping verify the encoding worked correctly.

**Summary**

Handling categorical variables is essential because:
- **Machine learning algorithms require numerical input**
- **Label encoding converts strings to integers efficiently**
- **Proper encoding maintains data consistency between training and testing**
- **The choice of encoding method affects model performance**

The next step after categorical encoding is feature scaling, which ensures all numerical features are on similar scales for optimal model performance.

### 3.2 Feature Scaling

#### Prepare features and target


In [ ]:
# Prepare features and target

X_train = train_data[numerical_cols + categorical_cols]
y_train = train_data['binary_class']
X_test = test_data[numerical_cols + categorical_cols]
y_test = test_data['binary_class']

#### Scale numerical features


In [ ]:
# Scale numerical features
scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[numerical_cols] = scaler.fit_transform(X_train[numerical_cols])
X_test_scaled[numerical_cols] = scaler.transform(X_test[numerical_cols])

print("Feature scaling completed")
print(f"Training features shape: {X_train_scaled.shape}")
print(f"Test features shape: {X_test_scaled.shape}")

**Question**: Why do we need Feature Scaling? Explain the Training and test fuature shapes in the ouput of the above cell.

## Part 4: Machine Learning Model Implementation



### 4.1 Logistic Regression (Baseline Model)


#### Train Logistic Regression


In [ ]:
# Train Logistic Regression
print("Training Logistic Regression...")
lr_model = LogisticRegression(random_state=42, max_iter=1000)
lr_model.fit(X_train_scaled, y_train)

#### Make predictions


In [ ]:
# Make predictions
lr_pred = lr_model.predict(X_test_scaled)

### Evaluate performance


In [ ]:
# Evaluate performance
print("\n=== LOGISTIC REGRESSION RESULTS ===")
print(f"Accuracy: {accuracy_score(y_test, lr_pred):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, lr_pred, target_names=['Normal', 'Attack']))

### 4.2 Random Forest (Ensemble Method)
#### Train Random Forest


In [ ]:
# Train Random Forest
print("Training Random Forest...")
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train_scaled, y_train)


#### Make predictions


In [ ]:
# Make predictions
rf_pred = rf_model.predict(X_test_scaled)

#### Evaluate performance

In [ ]:
# Evaluate performance
print("\n=== RANDOM FOREST RESULTS ===")
print(f"Accuracy: {accuracy_score(y_test, rf_pred):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, rf_pred, target_names=['Normal', 'Attack']))

### 4.3 Feature Importance Analysis


#### Get feature importance from Random Forest


In [ ]:
# Get feature importance from Random Forest
feature_importance = pd.DataFrame({
    'feature': X_train_scaled.columns,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)


#### Plot top 15 most important features


In [ ]:
# Plot top 15 most important features
plt.figure(figsize=(10, 8))
top_features = feature_importance.head(15)
sns.barplot(data=top_features, y='feature', x='importance')
plt.title('Top 15 Most Important Features for Attack Detection')
plt.xlabel('Feature Importance')
plt.tight_layout()
plt.show()

print("Top 10 Most Important Features:")
print(feature_importance.head(10))


### Training with top 15 important features

In [ ]:
# After the feature importance analysis, add this experiment:

print("=== FEATURE SELECTION EXPERIMENT ===")

# Get top 15 features
top_15_features = feature_importance.head(15)['feature'].tolist()
print(f"Top 15 features: {top_15_features}")

# Create reduced feature sets
X_train_top15 = X_train_scaled[top_15_features]
X_test_top15 = X_test_scaled[top_15_features]

print(f"\nOriginal feature count: {X_train_scaled.shape[1]}")
print(f"Reduced feature count: {X_train_top15.shape[1]}")

# Train Random Forest with top 15 features
rf_top15 = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_top15.fit(X_train_top15, y_train)

# Make predictions
rf_top15_pred = rf_top15.predict(X_test_top15)

# Compare performance
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

print("\n=== PERFORMANCE COMPARISON ===")
print("Full Feature Set (41 features):")
print(f"  Accuracy: {accuracy_score(y_test, rf_pred):.4f}")
print(f"  Precision: {precision_score(y_test, rf_pred):.4f}")
print(f"  Recall: {recall_score(y_test, rf_pred):.4f}")
print(f"  F1-Score: {f1_score(y_test, rf_pred):.4f}")

print("\nTop 15 Features:")
print(f"  Accuracy: {accuracy_score(y_test, rf_top15_pred):.4f}")
print(f"  Precision: {precision_score(y_test, rf_top15_pred):.4f}")
print(f"  Recall: {recall_score(y_test, rf_top15_pred):.4f}")
print(f"  F1-Score: {f1_score(y_test, rf_top15_pred):.4f}")

# Calculate performance difference
acc_diff = accuracy_score(y_test, rf_top15_pred) - accuracy_score(y_test, rf_pred)
print(f"\nAccuracy difference: {acc_diff:+.4f}")

if acc_diff > 0:
    print("✅ Feature selection IMPROVED performance!")
elif acc_diff < -0.01:
    print("❌ Feature selection REDUCED performance significantly")
else:
    print("➖ Feature selection had minimal impact")

**Computational Efficiency**


In [ ]:
import time

rf_model_time = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_top15_time = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
# Compare training times
start = time.time()
rf_model_time.fit(X_train_scaled, y_train)
full_time = time.time() - start

start = time.time()
rf_top15_time.fit(X_train_top15, y_train)
reduced_time = time.time() - start

speedup = full_time / reduced_time
print(f"All Features: {full_time} \t Top 15 Features: {reduced_time}")
print(f"Training speedup: {speedup:.2f}x faster")

###Validation Check
**To confirm this isn't just random variation, let's verify by using Cross-validation:**


In [ ]:
# Check if improvement is statistically significant
import numpy as np
from sklearn.model_selection import cross_val_score

print("=== CROSS-VALIDATION VERIFICATION ===")

# 5-fold CV with full features
cv_scores_full = cross_val_score(
    RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    X_train_scaled, y_train, cv=5, scoring='accuracy'
)

# 5-fold CV with top 15 features
cv_scores_top15 = cross_val_score(
    RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    X_train_scaled[top_15_features], y_train, cv=5, scoring='accuracy'
)

print(f"Full features CV accuracy: {cv_scores_full.mean():.4f} ± {cv_scores_full.std():.4f}")
print(f"Top 15 features CV accuracy: {cv_scores_top15.mean():.4f} ± {cv_scores_top15.std():.4f}")

improvement = cv_scores_top15.mean() - cv_scores_full.mean()
print(f"Average improvement: {improvement:+.4f}")

**Question:** Describe the Cross-valication briefly.

### What's Happening: Cross-Validation vs Train/Test Split

You've discovered an important difference between **cross-validation on training data** and **evaluation on held-out test data**.

#### Your Previous Results (77-80%):
```python
# This used SEPARATE test data (never seen during training)
rf_pred = rf_model.predict(X_test_scaled)  # Using test set
accuracy = accuracy_score(y_test, rf_pred)  # 77-80%
```

#### Your CV Results (99.89%):
```python
# This uses ONLY training data, split internally
cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=5)  # ~99.89%
```

### Why Cross-Validation Shows Higher Performance

### 1. **Data Distribution Differences**
The NSL-KDD dataset has different attack distributions between train and test sets:



In [ ]:
# Check the distributions
print("Training set attack distribution:")
print(train_data['attack_type'].value_counts(normalize=True).head())

print("\nTest set attack distribution:")
print(test_data['attack_type'].value_counts(normalize=True).head())



**Likely findings:**
- Training set: Dominated by well-known attacks (easier to classify)
- Test set: More diverse attacks, including novel ones (harder to classify)

### 2. **Cross-Validation Uses Training Data Only**
When you do 5-fold CV:
```
Training Data Split:
Fold 1: Train on attacks A,B,C,D → Test on attacks A,B,C,D ✅ Easy!
Fold 2: Train on attacks A,B,C,D → Test on attacks A,B,C,D ✅ Easy!
...
```

The model sees similar attack patterns in both training and validation folds.

### 3. **True Test Set is Deliberately Harder**
```
Real Test Evaluation:
Train on attacks A,B,C,D → Test on attacks E,F,G,H ❌ Much harder!
```

The test set likely contains:
- Novel attack variants
- Different attack distributions  
- More sophisticated attacks

## Verify This Hypothesis

Let's check if this explains the difference:



In [ ]:
# Compare attack type distributions
print("=== DATASET ANALYSIS ===")

# Training set analysis
train_attacks = train_data['attack_type'].value_counts()
print("Top 5 attacks in TRAINING set:")
print(train_attacks.head())
print(f"Training set diversity: {len(train_attacks)} unique attack types")

# Test set analysis
test_attacks = test_data['attack_type'].value_counts()
print("\nTop 5 attacks in TEST set:")
print(test_attacks.head())
print(f"Test set diversity: {len(test_attacks)} unique attack types")

# Check for novel attacks in test set
train_attack_types = set(train_data['attack_type'].unique())
test_attack_types = set(test_data['attack_type'].unique())

novel_attacks = test_attack_types - train_attack_types
print(f"\nNovel attacks in test set (not in training): {len(novel_attacks)}")
if novel_attacks:
    print(f"Novel attacks: {list(novel_attacks)}")

# Distribution overlap
from scipy.stats import entropy
print(f"\nDistribution similarity (lower = more different):")
# This will show how different the distributions are

The data distribution analysis reveals **exactly** why you're seeing such different performance between cross-validation (99%) and test set evaluation (77%).

## The Key Insight: Different Attack Landscapes

### Training Set (What the model learns on):
```
normal       53.46%  ← Majority class
neptune      32.72%  ← Dominant attack (DoS)
satan         2.88%  ← Port scanning
ipsweep       2.86%  ← Network scanning  
portsweep     2.33%  ← Port scanning
```

### Test Set (What the model faces in reality):
```
normal          43.08%  ← Less normal traffic
neptune         20.66%  ← Much less neptune
guess_passwd     5.46%  ← NEW: Password attacks
mscan            4.42%  ← NEW: Mass scanning
warezmaster      4.19%  ← NEW: Unauthorized software
```

## Why This Creates the Performance Gap

### 1. **Domain Shift**
The model was trained primarily on:
- **Neptune attacks** (32.7% of training data)
- **Scanning attacks** (satan, ipsweep, portsweep)

But gets tested on:
- **Password attacks** (guess_passwd) - likely not in training
- **Mass scanning** (mscan) - different from training scanners
- **Software piracy** (warezmaster) - completely different attack class

### 2. **Class Imbalance Shift**




In [ ]:
# Let's verify this hypothesis
print("=== ATTACK TYPE OVERLAP ANALYSIS ===")

train_attacks = set(train_data['attack_type'].unique())
test_attacks = set(test_data['attack_type'].unique())

# Find attacks only in test set
novel_test_attacks = test_attacks - train_attacks
print(f"Attacks in TEST but not in TRAINING: {len(novel_test_attacks)}")
print(f"Novel attacks: {sorted(list(novel_test_attacks))}")

# Find attacks only in training set
training_only = train_attacks - test_attacks
print(f"\nAttacks in TRAINING but not in TEST: {len(training_only)}")
print(f"Training-only attacks: {sorted(list(training_only))}")

# Calculate how much of test set consists of novel attacks
test_novel_proportion = test_data[test_data['attack_type'].isin(novel_test_attacks)].shape[0] / test_data.shape[0]
print(f"\nProportion of test set with novel attacks: {test_novel_proportion:.2%}")

## This is Actually Brilliant Dataset Design!

### Why NSL-KDD Does This:
1. **Simulates Real Cybersecurity Environment**
   - Attackers constantly develop new techniques
   - Models must generalize beyond known attacks
   - Zero-day detection capability testing

2. **Prevents Dataset Overfitting**
   - Forces models to learn general patterns, not memorize specific attacks
   - More realistic performance estimates
   - Better evaluation of feature importance

3. **Tests True Machine Learning Capability**
   - Easy to get high accuracy on seen attacks
   - Hard to generalize to unseen attack patterns
   - Reveals which features capture fundamental malicious behavior

## Why Your Feature Selection Still Worked

Your +0.88% improvement on the test set is actually **more impressive** than it first appeared because:

### 1. **Better Generalization**
The top 15 features you selected capture **fundamental network attack patterns** that transfer across different attack types:

```python
# Your top features (generalize well):
'dst_bytes', 'src_bytes'           # Data volume patterns
'same_srv_rate', 'diff_srv_rate'   # Service access patterns  
'dst_host_srv_count'               # Host behavior patterns
'flag'                             # Connection state patterns
```

These features work for **both** neptune (training) **and** guess_passwd (test) attacks!

### 2. **Reduced Overfitting to Training Attacks**
By removing the less important features, you prevented the model from over-learning neptune-specific patterns that don't apply to password attacks.

## Cybersecurity Lesson

This perfectly illustrates a core cybersecurity challenge:

### The Training vs Reality Gap:
```
Training Environment:
- Known attack signatures
- Familiar attack patterns  
- High detection rates (99%)

Production Environment:  
- Novel attack variants
- Evolving threat landscape
- Lower detection rates (77%)
```

### Why 77% is Actually Good:
- Detecting 3 out of 4 attacks including novel ones
- Much better than signature-based systems (would catch ~0% of novel attacks)
- Room for improvement with more sophisticated techniques

## Conclusion

Your results reveal **perfect machine learning behavior**:

1. ✅ **High CV performance** shows the model learns training patterns well
2. ✅ **Lower test performance** shows realistic generalization capability  
3. ✅ **Feature selection improvement** demonstrates better generalization
4. ✅ **Attack distribution analysis** explains the performance gap

This is exactly what you want to see in a cybersecurity ML system - good performance on known threats AND some capability against novel attacks. Your feature selection made the model more robust to new, unseen attack patterns!

**This makes for an excellent teaching example** about the realities of deploying ML in cybersecurity environments.


## Why This Matters for Cybersecurity

### 1. **Realistic Evaluation**
```python
# Cross-validation result (overly optimistic)
print("CV suggests we catch 99.89% of attacks! 🎉")

# Test set result (realistic)  
print("Reality: we catch 77% of attacks 😅")
```

### 2. **Zero-Day Attack Detection**
- **Training data**: Known attack patterns (signatures)
- **Test data**: Includes novel attacks the model has never seen
- **Real world**: New attacks appear constantly

### 3. **Why NSL-KDD is Designed This Way**
The dataset creators intentionally made the test set harder to:
- Simulate real-world conditions
- Test generalization ability
- Prevent overly optimistic results

## The Correct Interpretation

### Your Feature Selection Results:
```python
# Cross-validation (internal to training data)
Full features: 99.89% → Top 15: 99.85% = -0.04% difference

# Test set evaluation (realistic performance)  
Full features: 76.66% → Top 15: 77.55% = +0.88% improvement ✅
```

**The test set results (77-80%) are the ones that matter!**

## Lesson for Your Students

This is a perfect teaching moment about:

### 1. **Data Leakage Prevention**
```python
# WRONG: CV includes data from same distribution
cv_score = cross_val_score(model, all_data, all_labels)

# RIGHT: Evaluate on truly held-out data
model.fit(X_train, y_train)
score = model.score(X_test_unseen, y_test_unseen)
```

### 2. **Cross-Validation vs Final Evaluation**
- **Cross-validation**: For model selection and hyperparameter tuning
- **Test set**: For final, unbiased performance estimate

### 3. **Cybersecurity Reality**
- Models perform well on known attack patterns
- Performance drops significantly on novel attacks
- Need robust evaluation that simulates real threats

## Important Note: CV vs Test Set Performance

You may notice that cross-validation shows ~99% accuracy while test set shows ~77%.
This is EXPECTED and reveals important insights:

1. **Cross-validation (99%)**: Model performance on similar data patterns
2. **Test set (77%)**: Model performance on novel/different attacks

The test set performance is the realistic measure of how your model would
perform against new, previously unseen cyber attacks in the real world.

This difference highlights why cybersecurity ML is challenging - attackers
constantly evolve their techniques to evade detection!


## Conclusion

Your observation is **spot-on** and reveals a fundamental truth about machine learning evaluation. The 77-80% accuracy on the test set is the realistic performance measure, while the 99% CV score shows how well the model learns patterns within the training data distribution.

This makes your +0.88% improvement from feature selection even more impressive - you improved performance on the harder, more realistic test scenario!

###Advanced Feature Selection Experiment
**For a more thorough analysis, test multiple feature counts:**


In [ ]:
# Test different numbers of top features
feature_counts = [5, 10, 15, 20, 25, 30, 35, 41]
results = []

for n_features in feature_counts:
    # Select top n features
    top_features = feature_importance.head(n_features)['feature'].tolist()

    # Train model
    X_train_subset = X_train_scaled[top_features]
    X_test_subset = X_test_scaled[top_features]

    rf_subset = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    rf_subset.fit(X_train_subset, y_train)

    # Evaluate
    pred_subset = rf_subset.predict(X_test_subset)
    accuracy = accuracy_score(y_test, pred_subset)

    results.append({
        'n_features': n_features,
        'accuracy': accuracy
    })

    print(f"Top {n_features} features: Accuracy = {accuracy:.4f}")

# Plot results
import matplotlib.pyplot as plt
import pandas as pd

results_df = pd.DataFrame(results)
plt.figure(figsize=(10, 6))
plt.plot(results_df['n_features'], results_df['accuracy'], marker='o')
plt.xlabel('Number of Top Features Used')
plt.ylabel('Accuracy')
plt.title('Model Performance vs Number of Features')
plt.grid(True)
plt.show()

# Find optimal number of features
best_idx = results_df['accuracy'].idxmax()
optimal_features = results_df.loc[best_idx, 'n_features']
best_accuracy = results_df.loc[best_idx, 'accuracy']

print(f"\nOptimal number of features: {optimal_features}")
print(f"Best accuracy achieved: {best_accuracy:.4f}")

### Save the Trained Models

In [ ]:
from datetime import datetime
import json
import joblib

# Complete model saving for cybersecurity lab
def save_cybersec_models():
    """Save all models and preprocessing objects from the cybersecurity lab"""

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    model_dir = f"cybersec_models_{timestamp}"
    os.makedirs(model_dir, exist_ok=True)

    # Save models
    joblib.dump(lr_model, f'{model_dir}/logistic_regression.pkl')
    joblib.dump(rf_model, f'{model_dir}/random_forest.pkl')

    # Save preprocessing
    joblib.dump(scaler, f'{model_dir}/feature_scaler.pkl')
    joblib.dump(label_encoders, f'{model_dir}/label_encoders.pkl')

    # Save feature information
    feature_info = {
        'feature_names': list(X_train_scaled.columns),
        'categorical_columns': categorical_cols,
        'numerical_columns': numerical_cols,
        'n_features': X_train_scaled.shape[1]
    }

    # Save performance metrics
    lr_pred = lr_model.predict(X_test_scaled)
    rf_pred = rf_model.predict(X_test_scaled)

    performance = {
        'logistic_regression': {
            'accuracy': accuracy_score(y_test, lr_pred),
            'precision': precision_score(y_test, lr_pred),
            'recall': recall_score(y_test, lr_pred),
            'f1_score': f1_score(y_test, lr_pred)
        },
        'random_forest': {
            'accuracy': accuracy_score(y_test, rf_pred),
            'precision': precision_score(y_test, rf_pred),
            'recall': recall_score(y_test, rf_pred),
            'f1_score': f1_score(y_test, rf_pred)
        },
        'dataset_info': {
            'train_samples': X_train_scaled.shape[0],
            'test_samples': X_test_scaled.shape[0],
            'attack_distribution': train_data['attack_type'].value_counts().to_dict()
        },
        'timestamp': timestamp
    }

    # Save metadata
    with open(f'{model_dir}/model_info.json', 'w') as f:
        json.dump({
            'feature_info': feature_info,
            'performance': performance
        }, f, indent=2, default=str)

    print(f"✅ Complete model package saved to: {model_dir}")
    print(f"📊 LR Accuracy: {performance['logistic_regression']['accuracy']:.4f}")
    print(f"📊 RF Accuracy: {performance['random_forest']['accuracy']:.4f}")

    return model_dir

# Save models
saved_dir = save_cybersec_models()

## Part 5: Model Evaluation and Visualization


### 5.1 Confusion Matrix


### What is a Confusion Matrix?

A **confusion matrix** is a table used to evaluate the performance of a classification model. It shows the actual vs predicted classifications in a structured format, making it easy to see where the model is making correct predictions and where it's getting confused.

**Basic Structure**

For a **binary classification** problem (like Normal vs Attack traffic):

```
                    PREDICTED
                 Normal  Attack
        Normal  [  TN      FP  ]
ACTUAL  
        Attack  [  FN      TP  ]
```

**Key Terms:**
- **TP (True Positive)**: Correctly predicted attacks
- **TN (True Negative)**: Correctly predicted normal traffic  
- **FP (False Positive)**: Normal traffic wrongly classified as attack
- **FN (False Negative)**: Attacks wrongly classified as normal traffic

**Example from Cybersecurity**

Let's say we tested our model on 1000 network connections:

```
                    PREDICTED
                 Normal  Attack
        Normal  [  850     50  ]   900 actual normal
ACTUAL  
        Attack  [   20     30  ]   50 actual attacks
                   870    80      Total: 1000
```

**Reading the Matrix:**
- **850**: Normal traffic correctly identified as normal ✅
- **30**: Attacks correctly identified as attacks ✅  
- **50**: Normal traffic wrongly flagged as attacks ❌ (False Alarms)
- **20**: Attacks that went undetected ❌ (Missed Threats)

**Calculating Performance Metrics**
From the confusion matrix, we can calculate important metrics:

##### 1. **Accuracy** = (TP + TN) / Total
```
Accuracy = (30 + 850) / 1000 = 0.88 = 88%
```
*Overall correctness of the model*

##### 2. **Precision** = TP / (TP + FP)
```
Precision = 30 / (30 + 50) = 0.375 = 37.5%
```
*Of all predicted attacks, how many were actually attacks?*

##### 3. **Recall (Sensitivity)** = TP / (TP + FN)
```
Recall = 30 / (30 + 20) = 0.60 = 60%
```
*Of all actual attacks, how many did we catch?*

##### 4. **Specificity** = TN / (TN + FP)
```
Specificity = 850 / (850 + 50) = 0.944 = 94.4%
```
*Of all normal traffic, how many were correctly identified?*

#### Visual Representation

Here's how to interpret a confusion matrix visually:

```
🎯 PERFECT PREDICTIONS (Diagonal)
┌─────────────────────────────────┐
│  ✅ TN        ❌ FP             │  
│  (Good)       (False Alarms)    │
│                                 │
│  ❌ FN        ✅ TP             │
│  (Missed)     (Good)            │
└─────────────────────────────────┘

GOAL: Maximize diagonal values (TN + TP)
      Minimize off-diagonal values (FP + FN)
```
The confusion matrix is your best friend for understanding exactly how your cybersecurity model is performing and where it needs improvement! 🛡️📊

#### Now, Let's check the Confusion Matrixes of our trained models.

In [ ]:
#import libraries
from sklearn.metrics import confusion_matrix
import seaborn as sns

#### Create confusion matrices


In [ ]:
# Create confusion matrices
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Logistic Regression Confusion Matrix

cm_lr = confusion_matrix(y_test, lr_pred)
sns.heatmap(cm_lr, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Normal', 'Attack'],
            yticklabels=['Normal', 'Attack'], ax=axes[0])
axes[0].set_title('Logistic Regression Confusion Matrix')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

# Random Forest Confusion Matrix
cm_rf = confusion_matrix(y_test, rf_pred)
sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Normal', 'Attack'],
            yticklabels=['Normal', 'Attack'], ax=axes[1])
axes[1].set_title('Random Forest Confusion Matrix')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')

plt.tight_layout()
plt.show()

### 5.2 ROC Curve Analysis



#### What is ROC Curve?

**ROC** stands for **Receiver Operating Characteristic** curve. It's a graphical plot that illustrates the diagnostic ability of a binary classifier by showing the trade-off between sensitivity (True Positive Rate) and specificity (False Positive Rate) at various threshold settings.

**Basic Components**

**The Axes:**
- **X-axis**: False Positive Rate (FPR) = FP / (FP + TN)
- **Y-axis**: True Positive Rate (TPR) = TP / (TP + FN) = Recall/Sensitivity

**The Curve:**
- Shows performance at **all possible thresholds**
- Each point represents a different classification threshold
- Helps choose optimal threshold for your use case

**Visual Understanding**

```
TPR (Sensitivity)
    ↑
1.0 ┤     ●← Perfect Classifier (0,1)
    │    ╱
0.8 ┤   ╱  ← Good Model
    │  ╱
0.6 ┤ ╱     ← Our Model
    │╱
0.4 ┤      ← Poor Model  
    │
0.2 ┤ ╱ ← Random Classifier
    │╱
0.0 ┼─────────────────────→ FPR
   0.0  0.2  0.4  0.6  0.8  1.0
```

#### How ROC Curve Works

**The Threshold Concept:**
Most classifiers output **probabilities** (0.0 to 1.0), then apply a threshold to make final predictions:

```python
# Example predictions from your model:
probabilities = [0.1, 0.3, 0.6, 0.8, 0.9]
actual_labels = [ 0,   0,   1,   1,   1 ]

# Different thresholds give different results:
threshold = 0.5: predictions = [0, 0, 1, 1, 1] → Good balance
threshold = 0.3: predictions = [0, 1, 1, 1, 1] → More sensitive
threshold = 0.7: predictions = [0, 0, 0, 1, 1] → More specific
```

#### ROC Points Generation:
For each threshold, calculate TPR and FPR:

```
Threshold  Predictions   TP  FP  TN  FN    TPR    FPR
   0.1     [1,1,1,1,1]    3   2   0   0   1.000  1.000
   0.3     [0,1,1,1,1]    3   1   1   0   1.000  0.500  
   0.5     [0,0,1,1,1]    3   0   2   0   1.000  0.000
   0.7     [0,0,0,1,1]    2   0   2   1   0.667  0.000
   0.9     [0,0,0,0,1]    1   0   2   2   0.333  0.000
   1.0     [0,0,0,0,0]    0   0   2   3   0.000  0.000
```


#### **AUC (Area Under the Curve)**

**What is AUC?**
- **Area Under the ROC Curve**
- Single number summarizing model performance
- Range: 0.0 to 1.0

##### AUC Interpretation:
```
AUC = 1.0  → Perfect classifier 🎯
AUC = 0.9  → Excellent model ⭐⭐⭐⭐⭐
AUC = 0.8  → Good model ⭐⭐⭐⭐
AUC = 0.7  → Fair model ⭐⭐⭐
AUC = 0.6  → Poor model ⭐⭐
AUC = 0.5  → Random guessing (useless) ❌
AUC < 0.5  → Worse than random (flip predictions!) 🤮
```

#### Cybersecurity Context

##### Real-World Example:
Your intrusion detection system outputs attack probabilities:

```
Connection 1: 0.95 probability → Definitely attack
Connection 2: 0.78 probability → Likely attack  
Connection 3: 0.45 probability → Unsure
Connection 4: 0.12 probability → Likely normal
Connection 5: 0.03 probability → Definitely normal
```

##### **Different Thresholds, Different Security Postures:**

###### **High Security (Threshold = 0.3)**
```
✅ Catches almost all attacks (high sensitivity)
❌ Many false alarms (low specificity)
👮 Paranoid security guard - stops everyone suspicious
```

###### **Balanced Security (Threshold = 0.5)**
```
⚖️ Balance between catching attacks and false alarms
📊 Often the default choice
🎯 Moderate security guard - reasonable screening
```

###### **Low False Alarms (Threshold = 0.8)**
```
✅ Very few false alarms (high specificity)  
❌ Might miss subtle attacks (lower sensitivity)
😴 Relaxed security guard - only stops obvious threats
```





#### **Why is it Called "Area" Under the Curve?**
The term "area" is used literally - it refers to the actual **geometric area** of the region under the ROC curve when plotted on a graph.

#### **Interpreting ROC Curves**

##### Curve Shapes and Their Meanings: Visual Understanding

```
TPR ↑
1.0 ┤     ●
    │    ╱█
0.8 ┤   ╱██  ← This shaded region
    │  ╱███     is the "AREA"
0.6 ┤ ╱████
    │╱█████
0.4 ┤██████
    │███████
0.2 ┤████████
    │█████████
0.0 ┼─────────→ FPR
   0.0       1.0

AUC = Shaded Area = 0.85
```

## Mathematical Perspective

### It's a Real Geometric Area!

The ROC curve creates a shape on a 1×1 square plot:
- **X-axis**: 0 to 1 (False Positive Rate)
- **Y-axis**: 0 to 1 (True Positive Rate)
- **Total possible area**: 1 × 1 = 1.0

### Why Area Matters:

```python
# The area tells us: "What fraction of the unit square
# is covered by our model's performance?"

Perfect Model (AUC = 1.0):
┌─●─┐  ← Covers entire upper area
│███│
│███│  Area = 1.0 (100% of square)
│███│
└───┘

Random Model (AUC = 0.5):
┌───┐  ← Diagonal splits square in half
│╱██│
│██╱│  Area = 0.5 (50% of square)
│██│
└───┘

Terrible Model (AUC = 0.0):
┌───┐  ← Covers lower area only
│   │
│   │  Area = 0.0 (0% of upper square)
│   │
└─●─┘
```



#### Key Insights for Cybersecurity

##### 1. **ROC vs Business Requirements**
- **High-security environments**: Accept higher FPR for lower FNR
- **Resource-constrained teams**: Need lower FPR to reduce alert fatigue
- **Compliance requirements**: May mandate minimum detection rates

##### 2. **Class Imbalance Considerations**
```python
# In cybersecurity datasets, attacks are often rare
normal_traffic = 95%    # Majority class
attack_traffic = 5%     # Minority class

# ROC curves can be optimistic with imbalanced data
# Also consider Precision-Recall curves for better insight
```

##### 3. **Operational Thresholds**
```python
# Common cybersecurity threshold strategies:

# Conservative (High Security)
threshold = 0.3  # Catch more attacks, more false alarms

# Balanced (Default)  
threshold = 0.5  # Standard ML threshold

# Aggressive (Low Noise)
threshold = 0.7  # Fewer false alarms, might miss subtle attacks
```

#### ROC vs Other Metrics

| Metric | Best For | Limitation |
|--------|----------|------------|
| **ROC/AUC** | Overall model comparison | Can be misleading with imbalanced data |
| **Precision-Recall** | Imbalanced datasets | Less intuitive than ROC |
| **Confusion Matrix** | Understanding specific errors | Single threshold view |
| **F1-Score** | Balancing precision/recall | Single number summary |

#### Summary

ROC Curve Analysis helps you:

1. **📊 Visualize Performance**: See how your model performs across all thresholds
2. **⚖️ Choose Optimal Threshold**: Based on your security requirements
3. **🔄 Compare Models**: AUC provides single metric for comparison
4. **💰 Make Business Decisions**: Balance security vs operational costs
5. **🎯 Understand Trade-offs**: Between catching attacks and false alarms

In cybersecurity, ROC curves are essential for finding the right balance between **catching bad guys** (high TPR) and **not crying wolf** (low FPR)! 🛡️📈

### **Now, let's see our ROC/AUC curve**

In [ ]:
#import libraries
from sklearn.metrics import roc_curve, auc

#### Get prediction probabilities


In [ ]:
# Get prediction probabilities
lr_proba = lr_model.predict_proba(X_test_scaled)[:, 1]
rf_proba = rf_model.predict_proba(X_test_scaled)[:, 1]

##### Calculate ROC curves


In [ ]:
# Calculate ROC curves
fpr_lr, tpr_lr, threshold_lr = roc_curve(y_test, lr_proba)
fpr_rf, tpr_rf, threshold_rf = roc_curve(y_test, rf_proba)


#### Calculate AUC


In [ ]:
# Calculate AUC
auc_lr = auc(fpr_lr, tpr_lr)
auc_rf = auc(fpr_rf, tpr_rf)

#### Plot ROC curves


In [ ]:
# Plot ROC curves
plt.figure(figsize=(8, 6))
plt.plot(fpr_lr, tpr_lr, label=f'Logistic Regression (AUC = {auc_lr:.3f})')
plt.plot(fpr_rf, tpr_rf, label=f'Random Forest (AUC = {auc_rf:.3f})')
plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves for Attack Detection Models')
plt.legend()
plt.grid(True)
plt.show()

print(f"Logistic Regression AUC: {auc_lr:.4f}")
print(f"Random Forest AUC: {auc_rf:.4f}")


### Question: Discuss about the ROC curve that you obtained.

## Part 6: Multi-class Classification
In the previous section, we only considered binary classification, i.e., attack or normal classification. In Part 6, we will explore multi-class classification.

### 6.1 Detecting Specific Attack Types


#### Create multi-class labels (top 5 attack types + normal)


In [ ]:
# Create multi-class labels (top 5 attack types + normal)
top_attacks = train_data['attack_type'].value_counts().head(5).index.tolist()
if 'normal' not in top_attacks:
    top_attacks.append('normal')


#### Filter data for multi-class classification


In [ ]:
# Filter data for multi-class classification
train_multi = train_data[train_data['attack_type'].isin(top_attacks)].copy()
test_multi = test_data[test_data['attack_type'].isin(top_attacks)].copy()


#### Prepare features and labels


In [ ]:
# Prepare features and labels
X_train_multi = train_multi[numerical_cols + categorical_cols]
y_train_multi = train_multi['attack_type']
X_test_multi = test_multi[numerical_cols + categorical_cols]
y_test_multi = test_multi['attack_type']


#### Scale features


In [ ]:
# Scale features
X_train_multi_scaled = X_train_multi.copy()
X_test_multi_scaled = X_test_multi.copy()
X_train_multi_scaled[numerical_cols] = scaler.transform(X_train_multi[numerical_cols])
X_test_multi_scaled[numerical_cols] = scaler.transform(X_test_multi[numerical_cols])

print(f"Multi-class dataset shapes:")
print(f"Training: {X_train_multi_scaled.shape}")
print(f"Test: {X_test_multi_scaled.shape}")
print(f"Attack types: {y_train_multi.unique()}")

### 6.2 Train Multi-class Random Forest


#### Train multi-class Random Forest


In [ ]:
# Train multi-class Random Forest
print("Training multi-class Random Forest...")
rf_multi = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_multi.fit(X_train_multi_scaled, y_train_multi)

#### Make predictions


In [ ]:
%%time
# Make predictions
rf_multi_pred = rf_multi.predict(X_test_multi_scaled)

#### Evaluate performance

In [ ]:
# Evaluate performance
print("\n=== MULTI-CLASS RANDOM FOREST RESULTS ===")
print(f"Accuracy: {accuracy_score(y_test_multi, rf_multi_pred):.4f}")
print("\nClassification Report:")
print(classification_report(y_test_multi, rf_multi_pred))

#### Plot multi-class confusion matrix

In [ ]:
# Plot multi-class confusion matrix
plt.figure(figsize=(8, 6))
cm_multi = confusion_matrix(y_test_multi, rf_multi_pred)
sns.heatmap(cm_multi, annot=True, fmt='d', cmap='Blues',
            xticklabels=rf_multi.classes_,
            yticklabels=rf_multi.classes_)
plt.title('Multi-class Attack Detection Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.xticks(rotation=45)
plt.yticks(rotation=45)
plt.tight_layout()
plt.show()

## Part 7: Lab Questions and Exercises

### Discussion Questions
1. **Compare the performance of Logistic Regression vs Random Forest**: Which model performs better and why? Consider both accuracy and interpretability.

2. **Feature Importance**: Based on the feature importance analysis, which network characteristics are most indicative of malicious activity? How might this inform cybersecurity strategies?

3. **False Positives vs False Negatives**: In cybersecurity, which type of error is more costly? How would you adjust the model to minimize the more critical error type?

4. **Multi-class Performance**: How does the multi-class classification performance compare to binary classification? What challenges arise when detecting specific attack types?


### Coding Exercises

**Exercise 1**: Implement a Decision Tree classifier and compare its performance to the existing models.


In [ ]:
from sklearn.tree import DecisionTreeClassifier

# Your code here
dt_model = DecisionTreeClassifier(random_state=42, max_depth=10)

# Train and evaluate the model
dt_model.fit(X_train_scaled, y_train)

# Make predictions
dt_pred_basic = dt_model.predict(X_test_scaled)
dt_proba_basic = dt_model.predict_proba(X_test_scaled)[:, 1]

# Evaluate performance
dt_accuracy = accuracy_score(y_test, dt_pred_basic)

print(f"✅ Basic Decision Tree Results:")
print(f"   Accuracy: {dt_accuracy:.4f}")
print(f"   Tree Depth: {dt_model.get_depth()}")
print(f"   Number of Leaves: {dt_model.get_n_leaves()}")

print("\n📊 Detailed Classification Report:")
print(classification_report(y_test, dt_pred_basic, target_names=['Normal', 'Attack']))

**Exercise 2**: Plot the Confusion Matrix and ROC curve


In [ ]:
print("\n📊 Decision Tree Confusion Matrix Analysis")
print("=" * 50)

# Calculate confusion matrix
cm_dt = confusion_matrix(y_test, dt_pred_basic)

# Plot confusion matrix
plt.figure(figsize=(6, 4))
sns.heatmap(cm_dt, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Normal', 'Attack'],
            yticklabels=['Normal', 'Attack'],
            cbar_kws={'label': 'Count'})
plt.title('Decision Tree Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()


# Ultra-simple ROC plot
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt

# Calculate and plot ROC
fpr, tpr, _ = roc_curve(y_test, dt_proba_basic)
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, label=f'AUC = {auc(fpr, tpr):.3f}')
plt.plot([0, 1], [0, 1], 'r--', label='Random')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend()
plt.show()

**Exercise 3**: Implement threshold tuning for the Random Forest model to optimize for different metrics (precision, recall, F1-score).


In [ ]:
from sklearn.metrics import precision_recall_curve

# Your code here
# Plot precision-recall curves and find optimal thresholds


## Part 8: Real-world Applications and Challenges

### 8.1 Deployment Considerations

When deploying ML models for cybersecurity in production environments, consider:

- **Real-time Processing**: Models must process network traffic in real-time
- **Concept Drift**: Attack patterns evolve, requiring model updates
- **Interpretability**: Security analysts need to understand why alerts are triggered
- **Scalability**: Handle high-volume network traffic
- **Integration**: Work with existing security infrastructure

### 8.2 Limitations and Ethical Considerations

- **Adversarial Attacks**: Attackers may try to evade ML detection
- **Privacy**: Monitoring network traffic raises privacy concerns
- **Bias**: Models may have biases based on training data
- **Interpretability**: Complex models may be difficult to interpret


## Lab Deliverables

Students should submit:
1. Completed Jupyter notebook with all code cells executed
2. Written responses to discussion questions
3. Results from coding exercises
4. Brief reflection on the challenges and potential improvements

## Extension Activities

For advanced tasks:
- Implement deep learning models (Neural Networks, LSTM)
- Explore unsupervised learning for anomaly detection
- Research adversarial machine learning techniques
- Implement ensemble methods combining multiple algorithms

## Resources for Further Learning

- **Books**: "Machine Learning for Cybersecurity" by Ankur Tyagi
- **Papers**: "A Survey of Machine Learning for Big Data Processing" in IEEE Access
- **Datasets**: CICIDS2017, UNSW-NB15, CTU-13
- **Tools**: Scikit-learn, TensorFlow, Keras, PyTorch